# Figure 2 — which single units follow the population state, and why

Figure 1 establishes a population anchoring state. This asks what individual
cells do inside it: which identities follow it, how putative interneurons
behave, and what distinguishes the cells that stay anchored while the
population does not.

## The interneuron call

There is no interneuron label in this dataset, so one is derived from the spike
waveform. The threshold is not a convention — peak-to-trough duration is fitted
with a two-component Gaussian mixture and the antimode taken. It is genuinely
bimodal: component means **0.246 and 0.690 ms, separated by 4.0 SD**, giving a
threshold of 0.42 ms. Putative fast-spiking cells are narrow-waveform **and**
in the top quartile of firing rate.

Had the distribution been unimodal the split would have been arbitrary, so the
separation is printed by the builder and shown in panel B rather than assumed.

## Why the open field is needed for the speed question

On the linear track speed and position covary strongly, so a cell that looks
spatially stable while the population is not could simply be tracking speed —
which does not stop when the map does. The open field breaks that confound:
position and speed are close to independent there, so the open-field speed
score is a clean read of speed modulation, measured in a different environment
from the one anchoring is defined in.

**The supplied GLM speed scores are not used.** Each session ships
`time_shifted_S_glms.parquet`, but its `median_score` is negative for nearly
every cell and only ~1% of entorhinal cells reach FDR significance — no dynamic
range, so a null against it would be uninterpretable. `of_speed_score.py`
computes r(firing rate, speed) in 200 ms bins over moving periods against a
circular-shift null instead:

| | supplied GLM | computed |
|---|---|---|
| cells beating null | ~1–4% | **51%** |
| OF1 vs OF2 reliability | — | **r = 0.75** |

Putative interneurons come out as the most speed-modulated group (+0.084),
which is the expected sanity check.

## Step 1 — build the per-cell table

Joins four sources on (mouse, day, cluster_id): the null-corrected agreement
with the population axis (`per_cell_pc1.csv`), the per-trial labels (for the
dissent measures), open-field identity and anatomy, spike waveform, and the
computed open-field speed score.

Two dissent measures, because *r* cannot distinguish them — a cell can be
anchored when the population is not, or non-anchored when the population is,
and both lower *r*:

- `stay` = P(cell anchored | population **not** anchored) — the dissenter
- `drop` = P(cell non-anchored | population anchored)

`stay` is the one the speed hypothesis is about.

Putative monosynaptic connections are detected here too, since panel H of the
main figure uses them. Detection is described in **S6**.


In [1]:
"""Per-cell table, the open-field speed score it depends on, and the
monosynaptic connections panel H draws on."""
import glob, os, subprocess, sys
FIGDIR = ('/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
          'AnchorDynamics2026')
PS = '/Users/harryclark/Documents/spatial-manifolds/data/population_state'
MS = '/Users/harryclark/Documents/spatial-manifolds/data/monosyn'

if not os.path.exists(f'{PS}/of_speed_score_w0.csv'):
    # ~10 min across two workers; one NWB load per open-field session
    for w in (0, 1):
        subprocess.run([sys.executable, f'{FIGDIR}/of_speed_score.py', str(w), '2'],
                       cwd=FIGDIR, check=True)
subprocess.run([sys.executable, f'{FIGDIR}/build_unit_table.py'], cwd=FIGDIR,
               check=True)

# panel H needs these, so they are built here rather than with the supplements;
# ~40 min over 3 workers on a cold start, a no-op once cached
if glob.glob(f'{MS}/connections2_w*.csv'):
    print('monosyn: cached')
else:
    _ps = [subprocess.Popen([sys.executable, f'{FIGDIR}/monosyn.py', 'batch',
                             str(w), '3'], cwd=FIGDIR) for w in range(3)]
    for _p in _ps:
        assert _p.wait() == 0


/Users/harryclark/Documents/spatial-manifolds/data/population_state/unit_table.csv: 12722 cells (cached; --rebuild to redo)
monosyn: cached


## Step 2 — the figure

- **A, B** two example sessions: the full MEC anchoring raster (cells that vary,
  in PC1 order, then the locked cells after a gap), PC1 of that raster, one
  example unit from each identity group, and two **locked** units. Every panel
  in a row shares the trial axis running down the page, and the dashed lines
  mark the **major population transitions** — the thresholded population state
  after a 9-trial median filter, computed exactly as in Figure 1, so that short
  flickers do not bury the raster under lines.
- **C** how the per-cell null is built, worked through on one real cell
- **D** agreement with the population axis as **excess over that null**, by
  identity, with mixed-model tests against chance
- **E** fraction of cells individually beating their own null, per mouse
- **F** what each identity does in the ±5 trials around a population switch
- **G** agreement **between** identities, as a 4 × 4 matrix over its own null
- **H** **locked cells** — which identities never leave one anchoring mode

The spike-timing counterpart to G — grid cells and interneurons are *wired*
together, not merely correlated — is **S1** rather than a panel here. It is a
second, optional line of support for a claim G already makes, and it arrives
with a detector that needs five panels of validation before its matrices can be
read.


In [2]:
"""Figure 2, built from the script so the notebook cannot drift from it."""
import runpy
from IPython.display import Image, display
_g = runpy.run_path(f'{FIGDIR}/fig2_single_units.py', run_name='__main__')
display(_g['fig'])

6455 entorhinal cells with an agreement score, 61 sessions, 8 mice


setup ready
plotting helpers ready
anchoring helper ready


setup ready
plotting helpers ready
anchoring helper ready


  excess over chance, LMM vs 0: grid +0.119 (p=0.002)  NGS -0.000 (p=0.33)  NS -0.023 (p=0.59)  int +0.080 (p=0.0004)


  grid-interneuron agreement +0.1086 (n=29 sessions, p=1.4e-06); other off-diagonal +0.0581
  locked: 24.8% of cells; interneuron always:never = 108:2, non-spatial = 39:293


saved /Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026/fig2_single_units.pdf


<Figure size 1120x1040 with 27 Axes>

## Results

**Identity matters, with grid cells and interneurons indistinguishable.**
Agreement with the population axis, tested with mixed models
(`r ~ group + (1|mouse) + (1|mouse:day)`, Holm-corrected) rather than over
cells — cells are nested in sessions in mice, and the cell-level tests were 4–8
orders of magnitude smaller:

| comparison | cell-level | **LMM** |
|---|---|---|
| grid vs non-grid spatial | 1.1×10⁻¹⁴ | 4.5×10⁻⁷ |
| grid vs non-spatial | 2.3×10⁻¹⁸ | 9.5×10⁻¹⁵ |
| **grid vs interneuron** | 0.045 | **0.84** |
| non-grid spatial vs non-spatial | 0.0013 | 5.2×10⁻⁵ |
| non-grid spatial vs interneuron | 2.5×10⁻⁸ | 7.7×10⁻⁷ |
| non-spatial vs interneuron | 8.8×10⁻¹³ | 3.3×10⁻¹⁴ |

Grid cells and putative interneurons track the population axis
**indistinguishably** (p = 0.84), and both clearly exceed the two non-grid
principal classes. The axis is followed by grid cells and local inhibition
together, not by spatial coding generally.

## Locked cells (row F) — switching sessions only

A cell can fail to agree with the population for two different reasons, and
*r* cannot tell them apart. It can vary and disagree, or it can **never vary at
all** — under the absolute gate a cell whose two clusters both fall on the same
side of its null is called anchored on every trial, or on none. Its correlation
with the population axis is then undefined: it appears as "no agreement" when
it never had the opportunity to agree.

**Row F is restricted to the 35 of 62 sessions with ≥10 trials in each state.**
In a single-state session every cell is locked by construction, which measures
how often a *session* is single-state, not how often a *cell* is. The
restriction strengthens the result rather than producing it.

Both example sessions' locked cells are non-grid spatial, which looks like a
finding and is not — NGS cells are ~70% of the population and lock at almost
exactly their base rate. The asymmetry is in the other two classes, running in
**opposite directions**:

| identity | always anchored | never anchored | ratio |
|---|---|---|---|
| putative interneuron | 109 | 2 | **55 : 1** |
| non-spatial | 39 | 293 | **1 : 7.5** |
| grid | — | — | ~1 : 1 |
| non-grid spatial | — | — | ~1 : 1 |

Interneurons lock **on**; non-spatial cells lock **off**. Panel F's second plot
shows this holds per mouse rather than in one animal.

## The speed test — a null, reported here rather than given panels

Dissenters (cells anchored while the population is not) are **not** the cells
that are speed-modulated in the open field:

| | top-decile dissenters | rest | p |
|---|---|---|---|
| OF speed score | +0.046 | +0.044 | **0.99** |
| speed-significant | 49% | 55% | — |

The pooled rank correlation is rho = −0.042 — significant only because
n = 4,217, trivially small, and in the *wrong* direction. Per mouse it is
inconsistent in sign. Note also that the pooled rho(follow, speed) = +0.066 at
p = 2×10⁻⁵ is **pseudoreplication**: per mouse it runs −0.213, −0.018, +0.059,
+0.005, −0.119, −0.016, +0.185.

---
# Supplements

Each answers one question. All are built from scripts in this folder, run below,
so the notebook and the PDFs cannot diverge.

## Data the supplements need

Two extra passes over the sessions, both cached:

- `build_trial_maps.py` — trial × position rate maps reduced to per-cell
  summaries (template correlations at zero and best offset, per-state rates).
  ~30 min across 3 workers.
- `trial_rate_speed.py` — per-trial firing rate and running speed, with a
  speed-matched split. Fast (no rate maps needed).
- `trial_map_null.py` — the chance level for the best-offset search, on a
  10-session subset. **Essential**: without it the S1 comparison reads as the
  opposite of what it is.

In [3]:
"""Cached inputs for the supplements. No-ops once built."""
import glob, os, subprocess, sys
PS = '/Users/harryclark/Documents/spatial-manifolds/data/population_state'

def ensure(pattern, script, workers=3):
    if glob.glob(f'{PS}/{pattern}'):
        print(f'{pattern}: cached'); return
    for w in range(workers):
        subprocess.run([sys.executable, f'{FIGDIR}/{script}', str(w), str(workers)],
                       cwd=FIGDIR, check=True)

ensure('trial_map_stats_w*.csv', 'build_trial_maps.py')
ensure('trial_rate_speed_w*.csv', 'trial_rate_speed.py')
if not os.path.exists(f'{PS}/trial_map_null.csv'):
    subprocess.run([sys.executable, f'{FIGDIR}/trial_map_null.py', '10'],
                   cwd=FIGDIR, check=True)
else:
    print('trial_map_null.csv: cached')

trial_map_stats_w*.csv: cached
trial_rate_speed_w*.csv: cached
trial_map_null.csv: cached


## S1 — what does a NON-ANCHORED trial look like for a cell?

Has the field moved, or gone? **Gone.**

The raw numbers look like displacement — zero-offset correlation collapses
(+0.33 → +0.07) while the best-over-offsets barely drops (+0.56 → +0.53). That
reading is wrong, and only the null shows it: **two unrelated real cells reach
+0.500** by the same best-of-100-offsets search. Non-anchored sits +0.029 above
that floor; anchored +0.062. The mean best offset on non-anchored trials is
47 cm against the 50 cm expected from a uniform pick.

So non-anchored means **loss of spatial correspondence**, not a displaced map.

In [5]:
_s2 = runpy.run_path(f'{FIGDIR}/fig2_supp_nonanchored.py', run_name='__main__')
display(_s2['fig'])

6342 cells with a template and both states, 61 sessions
  zero offset : anchored +0.329  non-anchored +0.072  p=0
  best offset : anchored +0.562  non-anchored +0.529
  NULL best offset: other cell 0.500, bin-shuffled 0.189  (1637 cells, 10 sessions)
  headroom above the other-cell floor: anchored +0.062, non-anchored +0.029


saved /Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026/fig2_supp_nonanchored.pdf


<Figure size 980x300 with 4 Axes>

## S2 — does firing rate change with the anchoring state?

**No — and the instructive part is how that became clear.** On the same 6,517
cells:

| measure | log₂ ratio | p |
|---|---|---|
| spatially averaged, unmatched | **−0.090** | 5×10⁻⁶⁴ |
| time averaged, unmatched | **−0.028** | 0.004 |
| time averaged, **speed-matched** | **+0.036** | 1×10⁻¹⁴ |

All three are "significant". Two disagree in sign with the third. A rate read
off a rate map divides by **occupancy**, which is set by running speed; spikes
per running second does not. The two correlate at r = 0.96 but differ in mean
(6.84 vs 9.03 Hz), enough for a ~3% state difference to flip.

A ~3% effect that reverses under a change of denominator and a behavioural
control is not a rate effect. **Anchoring is about where cells fire, not how
much.** Panel D also shows the circularity control: splitting by each cell's own
label reverses the sign again, because a trial with more spikes earns the
anchored label.

In [6]:
_s3 = runpy.run_path(f'{FIGDIR}/fig2_supp_rate.py', run_name='__main__')
display(_s3['fig'])

9386 cells, 62 sessions, 8 mice
  speed unmatched 31.19 vs 33.46 cm/s; matched 30.55 vs 30.73
  spatially av. (unmatched)          log2 -0.0904  p=5e-64  n=6517
  time av. (unmatched)               log2 -0.0282  p=0.004  n=6517
  time av. SPEED-MATCHED             log2 +0.0362  p=1e-14  n=6517


saved /Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026/fig2_supp_rate.pdf


<Figure size 980x580 with 8 Axes>

## S3 — how reliably is a cell's agreement score measured?

**Very.** Odd/even split-half **r = 0.94**, Spearman-Brown corrected **0.97**,
and uniform across identities (grid 0.99, interneuron 0.98, NGS 0.97, NS 0.95).

This matters because unreliable *r* would attenuate every identity difference in
Figure 2 toward zero. It does not, and the similar reliability across groups
rules out "one identity is simply measured worse".

In [7]:
_s5 = runpy.run_path(f'{FIGDIR}/fig2_supp_reliability.py', run_name='__main__')
display(_s5['fig'])

8118 cells, 61 sessions
  split-half r = 0.939; Spearman-Brown corrected = 0.969
  grid                   n=  477  split-half 0.971  corrected 0.985
  non-grid spatial       n= 5508  split-half 0.937  corrected 0.967
  non-spatial            n= 1633  split-half 0.908  corrected 0.952
  putative interneuron   n=  500  split-half 0.953  corrected 0.976
saved /Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026/fig2_supp_reliability.pdf


<Figure size 860x300 with 3 Axes>

## S4 — is following organised anatomically?

**Yes, mediolaterally — and independently of identity.**

| axis | rho | LMM alone | **+ identity** |
|---|---|---|---|
| **medial–lateral** | −0.147 | 1.6×10⁻²⁵ | **1.6×10⁻¹⁶** |
| dorsal–ventral | −0.137 | 0.025 | 0.32 |
| probe depth | +0.045 | 0.013 | 0.24 |

Using Figure 3's own 3400 µm boundary, medial cells follow at **r = 0.205** vs
lateral **0.120**, and it holds *within every identity* — medial grid cells 0.313
vs lateral 0.156. So this is not the dorsal grid-cell enrichment showing through.

The dorsoventral gradient, by contrast, **disappears once identity is in the
model** — that one is an identity gradient.

Coordinates are taken from Figure 3 rather than guessed: `coord_SCs_x` is
mediolateral (magnitudes 2610–3774 µm), `coord_SCs_z` is anterior-posterior.

In [8]:
_s6 = runpy.run_path(f'{FIGDIR}/fig2_supp_anatomy.py', run_name='__main__')
display(_s6['fig'])

6454 cells with coordinates, 60 sessions, 7 mice

axis                       rho  LMM p (alone)  LMM p (+identity)
medial–lateral (µm)     -0.147       1.56e-25           1.55e-16
dorsal–ventral (µm)     -0.137         0.0247               0.32
probe depth (µm)        +0.045         0.0127              0.237



saved /Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026/fig2_supp_anatomy.pdf


<Figure size 960x300 with 4 Axes>